<a href="https://colab.research.google.com/github/tyagisidd2411/Attendance-management-system/blob/development/Attendance_management_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio

In [2]:
import sqlite3
from datetime import datetime

# Create database
conn = sqlite3.connect("attendance.db", check_same_thread=False)
cursor = conn.cursor()

# Student table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    roll_no TEXT UNIQUE NOT NULL,
    course TEXT
)
""")

# Attendance table
cursor.execute("""
CREATE TABLE IF NOT EXISTS attendance (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    roll_no TEXT NOT NULL,
    date TEXT NOT NULL,
    status TEXT NOT NULL
)
""")

conn.commit()

print("Database created successfully!")

Database created successfully!


In [3]:
import sqlite3
from datetime import datetime


# -------------------------------
# ADD STUDENT
# -------------------------------

def add_student(name, roll_no, course):

    if name.strip() == "" or roll_no.strip() == "":
        return "❌ Please enter Name and Roll Number."

    conn = sqlite3.connect("attendance.db")
    cursor = conn.cursor()

    try:

        cursor.execute(
            """
            INSERT INTO students (name, roll_no, course)
            VALUES (?, ?, ?)
            """,
            (name, roll_no, course)
        )

        conn.commit()
        message = "✅ Student added successfully!"

    except sqlite3.IntegrityError:

        message = "❌ Roll Number already exists."

    conn.close()

    return message


# -------------------------------
# MARK ATTENDANCE
# -------------------------------

def mark_attendance(roll_no, status):

    if roll_no.strip() == "":
        return "❌ Please enter Roll Number."

    conn = sqlite3.connect("attendance.db")
    cursor = conn.cursor()

    # Check student
    cursor.execute(
        "SELECT name FROM students WHERE roll_no = ?",
        (roll_no,)
    )

    student = cursor.fetchone()

    if student is None:

        conn.close()

        return "❌ Student not found. Please add the student first."

    date = datetime.now().strftime("%Y-%m-%d")

    cursor.execute(
        """
        INSERT INTO attendance (roll_no, date, status)
        VALUES (?, ?, ?)
        """,
        (roll_no, date, status)
    )

    conn.commit()
    conn.close()

    return f"✅ Attendance marked for {student[0]}: {status}"


# -------------------------------
# VIEW STUDENTS
# -------------------------------

def view_students():

    conn = sqlite3.connect("attendance.db")
    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT id, name, roll_no, course
        FROM students
        """
    )

    data = cursor.fetchall()

    conn.close()

    return data


# -------------------------------
# VIEW ATTENDANCE
# -------------------------------

def view_attendance():

    conn = sqlite3.connect("attendance.db")
    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT roll_no, date, status
        FROM attendance
        ORDER BY date DESC
        """
    )

    data = cursor.fetchall()

    conn.close()

    return data


print("All functions created successfully!")

All functions created successfully!


In [4]:
import gradio as gr


with gr.Blocks(title="Student Attendance Management System") as app:

    gr.Markdown(
        """
        # 🎓 Student Attendance Management System

        **Python + SQLite + Gradio**

        Manage students and attendance easily.
        """
    )


    # =========================================
    # ADD STUDENT
    # =========================================

    with gr.Tab("➕ Add Student"):

        gr.Markdown("### Student Registration")

        name = gr.Textbox(
            label="Student Name",
            placeholder="Enter student name"
        )

        roll_no = gr.Textbox(
            label="Roll Number",
            placeholder="Enter roll number"
        )

        course = gr.Textbox(
            label="Course",
            placeholder="Example: B.Tech CS"
        )

        add_button = gr.Button(
            "Add Student",
            variant="primary"
        )

        student_message = gr.Textbox(
            label="Result"
        )

        add_button.click(
            fn=add_student,
            inputs=[name, roll_no, course],
            outputs=student_message
        )


    # =========================================
    # MARK ATTENDANCE
    # =========================================

    with gr.Tab("📝 Mark Attendance"):

        gr.Markdown("### Mark Student Attendance")

        attendance_roll = gr.Textbox(
            label="Roll Number",
            placeholder="Enter roll number"
        )

        attendance_status = gr.Dropdown(
            choices=["Present", "Absent"],
            value="Present",
            label="Attendance Status"
        )

        attendance_button = gr.Button(
            "Mark Attendance",
            variant="primary"
        )

        attendance_message = gr.Textbox(
            label="Result"
        )

        attendance_button.click(
            fn=mark_attendance,
            inputs=[attendance_roll, attendance_status],
            outputs=attendance_message
        )


    # =========================================
    # VIEW STUDENTS
    # =========================================

    with gr.Tab("👨‍🎓 View Students"):

        gr.Markdown("### Registered Students")

        student_table = gr.Dataframe(
            headers=[
                "ID",
                "Name",
                "Roll Number",
                "Course"
            ],
            datatype=[
                "number",
                "str",
                "str",
                "str"
            ],
            label="Student List"
        )

        refresh_students = gr.Button(
            "🔄 Refresh Students"
        )

        refresh_students.click(
            fn=view_students,
            outputs=student_table
        )


    # =========================================
    # ATTENDANCE RECORDS
    # =========================================

    with gr.Tab("📊 Attendance Records"):

        gr.Markdown("### Attendance History")

        attendance_table = gr.Dataframe(
            headers=[
                "Roll Number",
                "Date",
                "Status"
            ],
            datatype=[
                "str",
                "str",
                "str"
            ],
            label="Attendance"
        )

        refresh_attendance = gr.Button(
            "🔄 Refresh Attendance"
        )

        refresh_attendance.click(
            fn=view_attendance,
            outputs=attendance_table
        )


# Launch application

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://781cde44644ee7b1c2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
